# Tool Use — Lab 2: Email Assistant Agent

A **LangChain** tool-calling agent over a **simulated FastAPI email backend**.
You give natural-language requests ("delete the Happy Hour email"); the LLM picks
the right tools and runs them.

**Before running:** start the FastAPI email server first (see `README.md`) and set
`M3_EMAIL_SERVER_API_URL` in `.env`. The tools in `src/email_tools.py` call that
server over HTTP.

Key idea of this lab: *the tools you expose define what the agent can do.*

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from dotenv import load_dotenv

from src import config, email_tools
from src.rendering import print_html
from src.agent import build_agent
from langchain_core.messages import ToolMessage

load_dotenv()

from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
langfuse_handler = CallbackHandler()


## 1. Sanity-check the backend

Make sure the server is up and the tools reach it: send a test email, fetch it back.

Because the email tools are now `@tool` objects (LangChain `StructuredTool`s), call
them directly with `.invoke({...})` and dict arguments. `reset_database` is a plain
admin helper, so it's still called normally.

In [ ]:
new = email_tools.send_email.invoke(
    {"recipient": "test@example.com", "subject": "Lunch plans", "body": "Shall we meet at noon?"}
)
content_ = email_tools.get_email.invoke({"email_id": new["id"]})
print_html(json.dumps(content_, indent=2), title="email_tools sanity check")

In [ ]:
email_tools.reset_database()

A small helper to print the agent's message trace: every tool call/result, then the
final assistant message.

In [ ]:
def show_trace(result):
    """Print each ToolMessage in the run, then the final assistant content."""
    for m in result["messages"]:
        if isinstance(m, ToolMessage):
            print_html(m.content, title=f"Tool output: {m.name}")
    print_html(result["messages"][-1].content, title="Final assistant message")

## 2. Let the agent orchestrate tools

Ask for a multi-step task. The agent should chain:
`search_unread_from_sender` → `mark_email_as_read` → `send_email`.

In [ ]:
agent = build_agent([
   email_tools.search_unread_from_sender,
   email_tools.list_unread_emails,
   email_tools.search_emails,
   email_tools.get_email,
   email_tools.mark_email_as_read,
   email_tools.send_email,
])
result = agent.invoke(
   {"messages": [{"role": "user", "content":
       "Check for unread emails from boss@email.com, mark them as read, "
       "and send a polite follow-up."}]},
   config={"configurable": {"thread_id": "email-1"}, "callbacks": [langfuse_handler]},
)
show_trace(result)

langfuse.flush()

## 3. Tools define capability — missing `delete_email`

Ask the agent to delete an email **without** giving it `delete_email`. It can reason
but cannot act — proof that the available tools bound the agent's abilities.

In [ ]:
agent = build_agent([
    email_tools.search_unread_from_sender,
    email_tools.list_unread_emails,
    email_tools.search_emails,
    email_tools.get_email,
    email_tools.mark_email_as_read,
    email_tools.send_email,
])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete alice@work.com email"}]},
    config={"configurable": {"thread_id": "no-delete"}, "callbacks": [langfuse_handler]},
)
show_trace(result)

langfuse.flush()

Now add `delete_email` and re-run — the agent can finish the task.

In [ ]:
agent = build_agent([
    email_tools.search_unread_from_sender,
    email_tools.list_unread_emails,
    email_tools.search_emails,
    email_tools.get_email,
    email_tools.mark_email_as_read,
    email_tools.send_email,
    email_tools.delete_email,
])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete alice@work.com email"}]},
    config={"configurable": {"thread_id": "with-delete"}, "callbacks": [langfuse_handler]},
)
show_trace(result)

langfuse.flush()

In [ ]:
agent = build_agent([
    email_tools.search_unread_from_sender,
    email_tools.list_unread_emails,
    email_tools.search_emails,
    email_tools.get_email,
    email_tools.mark_email_as_read,
    email_tools.send_email,
    email_tools.delete_email,
])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete the happy hour email"}]},
    config={"configurable": {"thread_id": "happy-hour"}, "callbacks": [langfuse_handler]},
)
show_trace(result)

langfuse.flush()